In [1]:
import pandas as pd

label_columns = [
    'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 
    'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 
    'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices'
]

def load_and_clean_chexpert(csv_path):
    # Load metadata
    df = pd.read_csv(csv_path)
    
    # Ensure numeric types and handle NaNs
    df[label_columns] = df[label_columns].apply(pd.to_numeric, errors='coerce').fillna(0).astype(float)
    
    # Apply U-Zeros policy: Replace all -1 (uncertain) with 0 (negative)
    df[label_columns] = df[label_columns].replace(-1.0, 0.0)
    
    # Standardize image paths for the Kaggle environment
    df['Path'] = df['Path'].str.replace('CheXpert-v1.0-small/', '', regex=False)
    df['Path'] = df['Path'].apply(lambda x: f"/kaggle/input/chexpert/{x}")
    
    # Return cleaned dataframe containing only the Path and the 14 labels
    return pd.concat([df['Path'], df[label_columns]], axis=1)

# Generate cleanly separated dataframes
train_df = load_and_clean_chexpert('/kaggle/input/datasets/ashery/chexpert/train.csv')
valid_df = load_and_clean_chexpert('/kaggle/input/datasets/ashery/chexpert/valid.csv')

# Create the 5% subset for your convergence test
sampled_train_df = train_df.sample(frac=0.05, random_state=42).reset_index(drop=True)

print(f"Full Training Set: {len(train_df)}")
print(f"Sampled Training Set (5%): {len(sampled_train_df)}")
print(f"Validation Set: {len(valid_df)}")

Full Training Set: 223414
Sampled Training Set (5%): 11171
Validation Set: 234


In [2]:
import os
import torch
import cv2
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

class CheXpertDistillationDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe
        
        # 1. Spatial transforms (Applied once so both models see the exact same geometry)
        self.spatial_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            # Add random spatial augmentations here if needed (e.g., RandomHorizontalFlip)
        ])
        
        # 2. EfficientNet Normalization (ImageNet stats)
        self.effnet_normalize = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        # 3. ViT Normalization ([-1, 1] stats)
        self.vit_normalize = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        raw_path = row['Path']
        
        if raw_path.startswith('CheXpert-v1.0-small/'):
            relative_path = raw_path.replace('CheXpert-v1.0-small/', '')
        elif raw_path.startswith('/kaggle/input/chexpert/'):
            relative_path = raw_path.replace('/kaggle/input/chexpert/', '')
        else:
            relative_path = raw_path
            
        relative_path = relative_path.lstrip('/')
        image_path = os.path.join('/kaggle/input/datasets/ashery/chexpert/', relative_path)
        
        # OpenCV memory fragmentation fix
        img_array = cv2.imread(image_path)
        img_array = cv2.cvtColor(img_array, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(img_array)
        
        label = torch.tensor(row[1:].values.astype(float), dtype=torch.float32)
        
        # Apply common geometry
        base_image = self.spatial_transform(image)
        
        # Branch normalizations natively
        effnet_image = self.effnet_normalize(base_image)
        vit_image = self.vit_normalize(base_image)
        
        return effnet_image, vit_image, label

# Initialize single datasets that handle both normalizations natively
train_dataset = CheXpertDistillationDataset(dataframe=train_df)
valid_dataset = CheXpertDistillationDataset(dataframe=valid_df)

In [3]:
from torch.utils.data import DataLoader

# train_dataset = CheXpertDataset(dataframe=dataset, transform=transform)
# valid_dataset = CheXpertDataset(dataframe=dataset_valid, transform=transform)

# Drastically increase batch size to saturate VRAM and reduce transfer overhead
BATCH_SIZE = 128

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE,              # Safe for GPU VRAM
    shuffle=True, 
    num_workers=0,               
    pin_memory=False,            
)

valid_loader = DataLoader(
    valid_dataset, 
    batch_size=BATCH_SIZE,       
    shuffle=False, 
    num_workers=0,               
    pin_memory=False             
)
# Example: Iterate over the DataLoader
for eff_images, vit_images, labels in train_loader:
    print("EfficientNET Image batch shape:", eff_images.shape)
    print("ViT Image batch shape:", vit_images.shape)
    print("Label batch shape:", labels.shape)
    break

for eff_images, vit_images, labels in valid_loader:
    print("EfficientNET Image batch shape:", eff_images.shape)
    print("ViT Image batch shape:", vit_images.shape)
    print("Label batch shape:", labels.shape)
    break

EfficientNET Image batch shape: torch.Size([128, 3, 224, 224])
ViT Image batch shape: torch.Size([128, 3, 224, 224])
Label batch shape: torch.Size([128, 14])
EfficientNET Image batch shape: torch.Size([128, 3, 224, 224])
ViT Image batch shape: torch.Size([128, 3, 224, 224])
Label batch shape: torch.Size([128, 14])


In [4]:
import torch
import torch.nn as nn
import torchvision

class FocalLossMultiLabel(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        return torchvision.ops.sigmoid_focal_loss(
            inputs=inputs,
            targets=targets,
            alpha=self.alpha,
            gamma=self.gamma,
            reduction=self.reduction
        )

class logit_kd_loss(nn.Module):
    def __init__(self, alpha=0.5, T=2.0, focal_alpha=0.25, focal_gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.T = T
        # Initialize the hard label loss (Student vs Ground Truth)
        self.focal_loss = FocalLossMultiLabel(alpha=focal_alpha, gamma=focal_gamma, reduction='mean')
        # Initialize the soft label loss (Student vs Teacher)
        self.bce_logits = nn.BCEWithLogitsLoss(reduction='mean')

    def forward(self, student_logits, teacher_logits, labels):
        # 1. Hard Loss computation
        hard_loss = self.focal_loss(student_logits, labels)

        # 2. Soft Loss computation
        # Soften the teacher probabilities using Temperature
        soft_targets = torch.sigmoid(teacher_logits / self.T)
        # Scale the student logits to match
        scaled_student_logits = student_logits / self.T
        
        soft_loss = self.bce_logits(scaled_student_logits, soft_targets)

        # 3. Combine and apply T^2 gradient correction
        total_loss = (1.0 - self.alpha) * hard_loss + self.alpha * soft_loss * (self.T ** 2)
        
        return total_loss
    

In [5]:
import timm
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Load pretrained EfficientNet-B0 from timm (Pre-trained on ImageNet by default)
# num_classes=14 automatically replaces the 1000-class ImageNet head with a 14-class head
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=14)

# unfreeze all layers
for param in model.parameters():
    param.requires_grad = True

# Unfreeze only the classifier head
for param in model.get_classifier().parameters():
    param.requires_grad = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# if torch.cuda.device_count() > 1:
#     print(f"Utilizing {torch.cuda.device_count()} GPUs for training")
#     model = nn.DataParallel(model)

# Define optimizer and scheduler 
optimizer = AdamW(model.parameters(), lr=1e-5) # because we do all the network we kick the lr down a lil 
scheduler = CosineAnnealingLR(optimizer, T_max=10)

criterion = logit_kd_loss()

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

In [6]:
from transformers import ViTForImageClassification

# Point this strictly to the DIRECTORY containing the .json and .safetensors files
# e.g., '/kaggle/input/vit-weights-epoch-N'
teacher_model_dir = '/kaggle/input/datasets/thageo/vit-full-label-weights/vit-chest-xray-full-labels-epoch-4'

# Hugging Face automatically parses the config.json and loads the .safetensors
teacher_model = ViTForImageClassification.from_pretrained(teacher_model_dir)

teacher_model.to(device)
teacher_model.eval()

# Lock the gradients to prevent VRAM exhaustion
for param in teacher_model.parameters():
    param.requires_grad = False

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

In [7]:
import gc
import torch
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score
import warnings

warnings.filterwarnings('ignore', category=UserWarning, module='sklearn.metrics')

def train_distillation_model(student_model, teacher_model, train_loader, valid_loader, optimizer, scheduler, criterion, device, epochs=5):
    scaler = torch.amp.GradScaler()

    # Class names for CheXpert Competition subset
    class_names = [
        'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 
        'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 
        'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices'
    ]

    # CRITICAL: Lock the teacher model entirely to prevent VRAM exhaustion
    teacher_model.eval()
    for param in teacher_model.parameters():
        param.requires_grad = False

    for epoch in range(epochs):
        print(f"\n--- Epoch {epoch + 1}/{epochs} ---")

        # =======================
        # TRAINING PHASE
        # =======================
        student_model.train()
        train_loss = 0.0

        for effnet_images, vit_images, labels in train_loader:
            effnet_images = effnet_images.to(device)
            vit_images = vit_images.to(device)
            labels = labels.to(device)

            # 1. TEACHER FORWARD PASS (No Gradients)
            with torch.no_grad():
                with torch.amp.autocast(device_type="cuda"):
                    teacher_logits = teacher_model(vit_images).logits

            # 2. STUDENT FORWARD PASS (Requires Gradients)
            with torch.amp.autocast(device_type="cuda"):
                student_logits = student_model(effnet_images)
                loss = criterion(student_logits, teacher_logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

            train_loss += loss.item()

            # Aggressive batch-level garbage collection
            del effnet_images, vit_images, labels, teacher_logits, student_logits, loss

        train_loss /= len(train_loader)
        print(f"Training Loss: {train_loss:.4f}")

        # Dump unreferenced RAM and VRAM before validation starts
        gc.collect()
        torch.cuda.empty_cache()

        # =======================
        # VALIDATION PHASE
        # =======================
        student_model.eval()
        valid_loss = 0.0
        
        all_labels = []
        all_preds = []

        with torch.no_grad():
            for effnet_images, vit_images, labels in valid_loader:
                effnet_images = effnet_images.to(device)
                vit_images = vit_images.to(device)
                labels = labels.to(device)

                with torch.amp.autocast(device_type="cuda"):
                    teacher_logits = teacher_model(vit_images).logits
                    student_logits = student_model(effnet_images)
                    loss = criterion(student_logits, teacher_logits, labels)

                valid_loss += loss.item()

                # Metrics are calculated solely on the Student's actual output probabilities
                probs = torch.sigmoid(student_logits)
                
                all_labels.append(labels.cpu().numpy())
                all_preds.append(probs.cpu().numpy())
                
                del effnet_images, vit_images, labels, teacher_logits, student_logits, loss, probs

        valid_loss /= len(valid_loader)
        
        all_labels = np.vstack(all_labels)
        all_preds = np.vstack(all_preds)

        # =======================
        # ROBUST METRICS CALCULATION
        # =======================
        per_class_roc_auc = []
        per_class_ap = []
        
        for i in range(len(class_names)):
            if len(np.unique(all_labels[:, i])) > 1:
                roc_score = roc_auc_score(all_labels[:, i], all_preds[:, i])
                ap_score = average_precision_score(all_labels[:, i], all_preds[:, i])
            else:
                roc_score = np.nan
                ap_score = np.nan
                
            per_class_roc_auc.append(roc_score)
            per_class_ap.append(ap_score)

        valid_roc_scores = [s for s in per_class_roc_auc if not np.isnan(s)]
        valid_ap_scores = [s for s in per_class_ap if not np.isnan(s)]
        
        macro_roc_auc = np.mean(valid_roc_scores) if valid_roc_scores else np.nan
        macro_ap = np.mean(valid_ap_scores) if valid_ap_scores else np.nan

        # Delete stacked numpy arrays
        del all_labels, all_preds
        gc.collect()
        
        print(f"Validation Loss: {valid_loss:.4f} | Macro ROC-AUC: {macro_roc_auc:.4f} | Macro Avg Precision: {macro_ap:.4f}")
        
        print(f"{'Class':<28} | {'ROC-AUC':<8} | {'Avg Precision':<13}")
        print("-" * 55)
        for i, name in enumerate(class_names):
            roc_str = f"{per_class_roc_auc[i]:.4f}" if not np.isnan(per_class_roc_auc[i]) else "NaN"
            ap_str = f"{per_class_ap[i]:.4f}" if not np.isnan(per_class_ap[i]) else "NaN"
            print(f"{name:<28} | {roc_str:<8} | {ap_str:<13}")

        # =======================
        # SCHEDULER & CHECKPOINTING
        # =======================
        if scheduler is not None:
            scheduler.step()

        save_path = f"/kaggle/working/student-efficientnet-b0-epoch-{epoch+1}.pth"
        print(f"\nSaving checkpoint to {save_path}...")
        torch.save(student_model.state_dict(), save_path)

In [8]:
train_distillation_model(
    student_model=model, 
    teacher_model=teacher_model, 
    train_loader=train_loader, 
    valid_loader=valid_loader, 
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion, 
    device=device, 
    epochs=5
)


--- Epoch 1/5 ---
Training Loss: 1.3781
Validation Loss: 1.3180 | Macro ROC-AUC: 0.6705 | Macro Avg Precision: 0.4267
Class                        | ROC-AUC  | Avg Precision
-------------------------------------------------------
No Finding                   | 0.8509   | 0.5204       
Enlarged Cardiomediastinum   | 0.5207   | 0.4895       
Cardiomegaly                 | 0.7548   | 0.5717       
Lung Opacity                 | 0.8282   | 0.8358       
Lung Lesion                  | 0.1202   | 0.0049       
Edema                        | 0.8306   | 0.5525       
Consolidation                | 0.8787   | 0.5226       
Pneumonia                    | 0.5113   | 0.0530       
Atelectasis                  | 0.7391   | 0.5754       
Pneumothorax                 | 0.5725   | 0.0980       
Pleural Effusion             | 0.8339   | 0.6384       
Pleural Other                | 0.5451   | 0.0093       
Fracture                     | NaN      | NaN          
Support Devices              | 0.7311   |